# Using Polars for Data Preprocessing and Analysis

## Overview

In this exercise, you will walk through a complete data preprocessing workflow using Python and Polars.

The goal is to take a raw dataset and transform it into a clean, structured form that is ready for analysis or modeling.

You will learn how to:

- Load a large CSV file and inspect its schema
- Check for missing values and decide how to handle them
- Convert a currency string to a number and combine date parts into a timestamp
- Save the cleaned data as a Parquet file
- Build flag columns, per-user aggregates, and window features
- Test a simple fraud rule against the fraud label

Each step is explained in detail so you can understand _why_ it is done, not just _how_.

## Import Libraries

We are using the Polars library for data manipulation due to its performance advantages, especially with larger datasets.


In [1]:
import polars as pl

## Read the Input CSV File

We will read the dataset with Polars and time how long the read takes. First, download the `credit_card_transactions-ibm_v2.csv` file from this [Kaggle page](https://www.kaggle.com/datasets/ealtman2019/credit-card-transactions) and place it in the same folder as your Jupyter notebook. The file is about 2.3 GB.

:::{caution} Memory

The table has about 24 million rows and takes several GB of memory once loaded. Close other memory-heavy programs before you run this notebook.

:::

### `polars.read_csv()`

Let's read the CSV file using Polars and measure the time taken.


In [2]:
%%time
df_polars = pl.read_csv("credit_card_transactions-ibm_v2.csv")
df_polars

CPU times: user 9.64 s, sys: 4.26 s, total: 13.9 s
Wall time: 11.1 s


User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
i64,i64,i64,i64,i64,str,str,str,i64,str,str,f64,i64,str,str
0,0,2002,9,1,"""06:21""","""$134.09""","""Swipe Transaction""",3527213246127876953,"""La Verne""","""CA""",91750.0,5300,null,"""No"""
0,0,2002,9,1,"""06:42""","""$38.48""","""Swipe Transaction""",-727612092139916043,"""Monterey Park""","""CA""",91754.0,5411,null,"""No"""
0,0,2002,9,2,"""06:22""","""$120.34""","""Swipe Transaction""",-727612092139916043,"""Monterey Park""","""CA""",91754.0,5411,null,"""No"""
0,0,2002,9,2,"""17:45""","""$128.95""","""Swipe Transaction""",3414527459579106770,"""Monterey Park""","""CA""",91754.0,5651,null,"""No"""
0,0,2002,9,3,"""06:23""","""$104.71""","""Swipe Transaction""",5817218446178736267,"""La Verne""","""CA""",91750.0,5912,null,"""No"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1999,1,2020,2,27,"""22:23""","""$-54.00""","""Chip Transaction""",-5162038175624867091,"""Merrimack""","""NH""",3054.0,5541,null,"""No"""
1999,1,2020,2,27,"""22:24""","""$54.00""","""Chip Transaction""",-5162038175624867091,"""Merrimack""","""NH""",3054.0,5541,null,"""No"""
1999,1,2020,2,28,"""07:43""","""$59.15""","""Chip Transaction""",2500998799892805156,"""Merrimack""","""NH""",3054.0,4121,null,"""No"""


### Benchmarking with Pandas

You can compare the time taken by Polars with that of pandas for reading the same CSV file. This notebook does not run pandas, but you can paste the code below into a new cell to try it.

```python
%%time
df_pandas = pd.read_csv("credit_card_transactions-ibm_v2.csv")
```

The execution time will vary based on your system, but Polars is generally expected to be much faster for larger datasets. On the author's desktop with 32 GB RAM and a 13th Gen Intel(R) Core(TM) i7-13700, Polars took approximately 1 second while Pandas took around 22 seconds. That's more than a 20x speedup!

`pandas` time result:

```
CPU times: total: 3 s
Wall time: 22.6 s
```

`polars` time result:

```
CPU times: total: 1.02 s
Wall time: 941 ms
```


## Preprocessing Steps


### Convert `"Amount"` column to a numeric type

The code below performs data type conversion on the `"Amount"` column. It utilizes Polars' expression API to remove "$" and "," characters via `replace_all()`, subsequently casting the cleaned values to `Float64` to overwrite the existing column with numeric data.


In [3]:
df_polars = df_polars.with_columns(
    pl.col("Amount")
    .str.replace_all(r"[$,]", "")  # remove $ (and commas, just in case)
    .cast(pl.Float64)
    .alias("Amount")
)

In [4]:
df_polars.head(3)

User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
i64,i64,i64,i64,i64,str,f64,str,i64,str,str,f64,i64,str,str
0,0,2002,9,1,"""06:21""",134.09,"""Swipe Transaction""",3527213246127876953,"""La Verne""","""CA""",91750.0,5300,null,"""No"""
0,0,2002,9,1,"""06:42""",38.48,"""Swipe Transaction""",-727612092139916043,"""Monterey Park""","""CA""",91754.0,5411,null,"""No"""
0,0,2002,9,2,"""06:22""",120.34,"""Swipe Transaction""",-727612092139916043,"""Monterey Park""","""CA""",91754.0,5411,null,"""No"""


The output confirms that the `"Amount"` column has been successfully converted to a `f64` (Float64) data type.


### Check the Schema

#### What is a Schema in Polars?

In Polars, the **schema** is a fixed mapping of column names to their specific data types (dtypes). Unlike some other data libraries, Polars is built on the Apache Arrow memory format, which requires every column to have a strictly defined type from the moment the DataFrame is initialized. This schema acts as a "contract" for your data; if you try to perform a string operation on a column that the schema defines as an integer, Polars will stop you immediately with a clear error. This strictness is a core reason why Polars is so fast and memory-efficient---it knows exactly how much space to allocate in memory and which CPU instructions to use before it even touches the data.

The fundamental difference between Polars and pandas lies in how they handle type inference and consistency:

- **Predictability vs. Flexibility:** Pandas is "lazy" and flexible with types, often defaulting to the generic `object` dtype if it encounters mixed data (like a column with both numbers and strings). This can lead to "Object" columns that swallow up memory and slow down your code. Polars, conversely, is "eager" about types; it does not allow mixed types in a single column.

- **The Schema Check:** In pandas, you often don't know a calculation will fail until the code has already been running for several minutes. Because Polars has a defined schema and a lazy execution engine, it can inspect your entire query plan and check the schema for type mismatches _before_ it actually processes a single row of data.

- **Null Handling:** While pandas historically used `NaN` (a float value) to represent missing numbers - which could accidentally change an integer column into a float column - Polars handles nulls using a separate bitmask. This ensures that your integers stay integers, keeping your schema stable throughout your pipeline.


In [5]:
df_polars.schema

Schema([('User', Int64),
        ('Card', Int64),
        ('Year', Int64),
        ('Month', Int64),
        ('Day', Int64),
        ('Time', String),
        ('Amount', Float64),
        ('Use Chip', String),
        ('Merchant Name', Int64),
        ('Merchant City', String),
        ('Merchant State', String),
        ('Zip', Float64),
        ('MCC', Int64),
        ('Errors?', String),
        ('Is Fraud?', String)])

The `df_polars.schema` output is a structured dictionary-like object that acts as the blueprint for your DataFrame. It maps every column name to its specific Polars data type (such as `String`, `Float64`, or `Int64`). Unlike a simple list of names, this schema is the source of truth for the Query Optimizer; it allows Polars to validate operations - like ensuring you aren't trying to add a number to a date - without having to scan the actual data. When you look at this output, you are seeing the strict definitions that Polars uses to allocate memory and plan the most efficient way to execute your code.


### Check for missing values

Before cleaning anything, check which columns have missing values. `null_count()` returns one row with the number of nulls in each column. `unpivot()` turns that row into a long table, and the filter keeps only the columns that have at least one null.

In [6]:
null_counts = df_polars.null_count().unpivot(
    variable_name="column", value_name="n_missing"
)

null_counts.filter(pl.col("n_missing") > 0)

column,n_missing
str,u32
"""Merchant State""",2720821
"""Zip""",2878135
"""Errors?""",23998469


Missing values are not always errors. Before you drop or fill them, ask _why_ they are missing. The code below groups the transactions with no `"Zip"` by the type of merchant.

In [7]:
(
    df_polars.filter(pl.col("Zip").is_null())
    .with_columns(
        pl.when(pl.col("Merchant City") == "ONLINE")
        .then(pl.lit("Online merchant"))
        .otherwise(pl.lit("Merchant outside the U.S."))
        .alias("merchant_type")
    )
    .group_by("merchant_type")
    .agg(pl.len().alias("n_missing_zip"))
    .sort("n_missing_zip", descending=True)
)

merchant_type,n_missing_zip
str,u32
"""Online merchant""",2720821
"""Merchant outside the U.S.""",157314


Only three columns have missing values, and each has a business reason:

- **`"Merchant State"` and `"Zip"`** are empty for online merchants (`"Merchant City"` is `"ONLINE"`), which have no physical location. `"Zip"` is also empty for merchants outside the U.S. For those rows, `"Merchant State"` holds the country name instead, and a U.S. zip code does not apply.
- **`"Errors?"`** is empty when the transaction had no error, which is true for about 98% of rows.

So we keep these nulls. Filling them with a made-up zip code or a placeholder error would hide real information, such as whether a purchase was made online. Dropping the rows would remove every online transaction, which is exactly where most of the fraud is, as we will see below.

### Create a timestamp column from the date time components

The code below merges fragmented date and time components into a single, unified Datetime column named `"Datetime"`. By using `pl.datetime()`, you are instructing Polars to reach into the individual "Year", "Month", and "Day" columns and combine them with hours and minutes extracted from the `"Time"` string. To get those specific time units, the code uses `.str.slice()` to "cut" the hour and minute portions out of the string and casts them to integers.

The `"Time"` column only records hours and minutes (for example, `"06:21"`), so `"Datetime"` has minute resolution. Keep this in mind later: two transactions in the same minute get exactly the same timestamp.

In [8]:
df_polars = df_polars.with_columns(
    pl.datetime(
        pl.col("Year"),
        pl.col("Month"),
        pl.col("Day"),
        pl.col("Time").str.slice(0, 2).cast(pl.Int32),
        pl.col("Time").str.slice(3, 2).cast(pl.Int32),
    ).alias("Datetime")
)

df_polars

User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?,Datetime
i64,i64,i64,i64,i64,str,f64,str,i64,str,str,f64,i64,str,str,datetime[μs]
0,0,2002,9,1,"""06:21""",134.09,"""Swipe Transaction""",3527213246127876953,"""La Verne""","""CA""",91750.0,5300,null,"""No""",2002-09-01 06:21:00
0,0,2002,9,1,"""06:42""",38.48,"""Swipe Transaction""",-727612092139916043,"""Monterey Park""","""CA""",91754.0,5411,null,"""No""",2002-09-01 06:42:00
0,0,2002,9,2,"""06:22""",120.34,"""Swipe Transaction""",-727612092139916043,"""Monterey Park""","""CA""",91754.0,5411,null,"""No""",2002-09-02 06:22:00
0,0,2002,9,2,"""17:45""",128.95,"""Swipe Transaction""",3414527459579106770,"""Monterey Park""","""CA""",91754.0,5651,null,"""No""",2002-09-02 17:45:00
0,0,2002,9,3,"""06:23""",104.71,"""Swipe Transaction""",5817218446178736267,"""La Verne""","""CA""",91750.0,5912,null,"""No""",2002-09-03 06:23:00
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1999,1,2020,2,27,"""22:23""",-54.0,"""Chip Transaction""",-5162038175624867091,"""Merrimack""","""NH""",3054.0,5541,null,"""No""",2020-02-27 22:23:00
1999,1,2020,2,27,"""22:24""",54.0,"""Chip Transaction""",-5162038175624867091,"""Merrimack""","""NH""",3054.0,5541,null,"""No""",2020-02-27 22:24:00
1999,1,2020,2,28,"""07:43""",59.15,"""Chip Transaction""",2500998799892805156,"""Merrimack""","""NH""",3054.0,4121,null,"""No""",2020-02-28 07:43:00


The primary benefit of this transformation is that it shifts your dataset from a static table into a time-series format. Instead of treating "Year" and "Month" as independent categories, Polars now understands the linear flow of time between transactions. This allows you to perform advanced chronological operations, such as calculating the time elapsed between purchases, sorting transactions by occurrence, or resampling the data to see fraud trends by hour.


### (Optional) Write to a Parquet File

Parquet is a columnar storage file format that is optimized for performance and efficiency. It allows for faster read and write operations compared to CSV, especially with larger datasets. By writing the cleaned DataFrame to a Parquet file, you can significantly reduce the time it takes to load the data in future analyses, as Parquet files are designed for efficient data retrieval and storage.


In [9]:
# Remove redundant date and time component columns
df_for_parquet = df_polars.drop(["Year", "Month", "Day", "Time"])

# Reorder columns to match the original CSV structure, with the new "Datetime" column in a logical position
first_columns = ["User", "Card", "Amount", "Datetime"]
df_for_parquet = df_for_parquet.select([*first_columns, pl.exclude(first_columns)])

df_for_parquet.head(2)

User,Card,Amount,Datetime,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
i64,i64,f64,datetime[μs],str,i64,str,str,f64,i64,str,str
0,0,134.09,2002-09-01 06:21:00,"""Swipe Transaction""",3527213246127876953,"""La Verne""","""CA""",91750.0,5300,null,"""No"""
0,0,38.48,2002-09-01 06:42:00,"""Swipe Transaction""",-727612092139916043,"""Monterey Park""","""CA""",91754.0,5411,null,"""No"""


Write to a parquet file using Polars' `write_parquet()` method. This will save the cleaned DataFrame in a more efficient format for future use.


In [10]:
df_for_parquet.write_parquet("credit_card_transactions-ibm_v2.parquet")

:::{tip} Columnar vs row-based storage

CSV files are row-oriented.

```plaintext
row1: a,b,c,d,e
row2: a,b,c,d,e
row3: a,b,c,d,e
```

To read **one column**, the engine must:

- read every row
- parse every value
- throw most of it away

On the other hand Parquet files are column-oriented.

```plaintext
col_a: a a a
col_b: b b b
col_c: c c c
col_d: d d d
col_e: e e e
```

If your query only needs `Amount` and `Merchant Name`:

- CSV parses everything.
- Parquet only reads those two columns.

This is why Parquet is dramatically faster for analytics.

:::


:::{note} Parquet's compression

### CSV compression is "dumb"

- You gzip the whole file
- You must decompress the entire thing to read any part
- No awareness of data types

### Parquet compression is "smart"

- Each column compressed separately
- Repeated values occupy extremely small space
- Integers, floats, timestamps all compress differently
- Engines can skip compressed blocks they are not queried

:::


### Payment method frequency analysis

The code below provides a high-level summary of the payment methods used across your credit card dataset. The `.value_counts()` method performs a frequency analysis on the `"Use Chip"` column, identifying every unique entry (Chip, Online, and Swipe transactions).


In [11]:
df_polars["Use Chip"].value_counts(sort=True)

Use Chip,count
str,u32
"""Swipe Transaction""",15386082
"""Chip Transaction""",6287598
"""Online Transaction""",2713220


Looking at the numbers, you can see that Swipe Transactions are the dominant payment method in this dataset with over 15 million entries, followed by Chip Transactions at roughly 6.3 million and Online Transactions at about 2.7 million. This breakdown is essential for understanding consumer behavior or detecting potential fraud patterns, as certain types of transactions (like "Online" vs. "Swipe") carry different risk profiles.


### Fraud distribution analysis

The code below shows the frequency of the `"Is Fraud?"` column to determine the distribution of legitimate versus fraudulent transactions within your dataset.


In [12]:
df_polars["Is Fraud?"].value_counts(sort=True)

Is Fraud?,count
str,u32
"""No""",24357143
"""Yes""",29757


The data reveals that the overwhelming majority of transactions - over 24.3 million - are flagged as "No", while only 29,757 are flagged as "Yes". While the fraudulent cases represent a very small fraction of the total volume (roughly 0.12%), identifying this minority is the primary goal of most credit card analytics. Understanding this ratio is a critical first step before building a machine learning model, as it tells you that you'll need specialized techniques, like oversampling or specific loss functions, to account for the rarity of fraud.


### Flag suspicious transactions based on amount

The code below performs feature engineering by adding three new boolean (`True`/`False`) columns to `df_polars`. By using `with_columns()` with a list of expressions, Polars efficiently evaluates these conditions in parallel. Specifically, it identifies refunds (amounts less than zero), large transactions (amounts exceeding $500), and online activity (where the "Use Chip" status matches "Online Transaction"). We assign the result back to `df_polars` so the new columns are available in later cells.

In [13]:
df_polars = df_polars.with_columns(
    [
        (pl.col("Amount") < 0).alias("is_refund"),
        (pl.col("Amount") > 500).alias("large_txn"),
        (pl.col("Use Chip") == "Online Transaction").alias("is_online"),
    ]
)

df_polars.select(["User", "Amount", "Use Chip", "is_refund", "large_txn", "is_online"]).head()

User,Amount,Use Chip,is_refund,large_txn,is_online
i64,f64,str,bool,bool,bool
0,134.09,"""Swipe Transaction""",false,false,false
0,38.48,"""Swipe Transaction""",false,false,false
0,120.34,"""Swipe Transaction""",false,false,false
0,128.95,"""Swipe Transaction""",false,false,false
0,104.71,"""Swipe Transaction""",false,false,false


These "flag" columns can be useful for both exploratory analysis and machine learning. Instead of writing complex filters repeatedly, you can now quickly segment your data. For example, to see if "large transactions" are more likely to be "fraudulent" or to calculate the total volume of "online" versus in-person sales. This step transforms raw data into behavioral features, providing the model or the analyst with clear, binary signals that highlight high-interest events within the transaction stream.


### Filter rows based on conditions

You can also apply a filter to isolate a very specific subset of your data: online refunds. Because the flags are now columns, the filter can use them directly. By passing multiple conditions into the `.filter()` method, Polars treats them as a logical AND operation, meaning a transaction will only remain in the resulting DataFrame if it is both a refund (`is_refund`) and an online transaction (`is_online`).

In [14]:
df_polars.filter(pl.col("is_refund"), pl.col("is_online"))

User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?,Datetime,is_refund,large_txn,is_online
i64,i64,i64,i64,i64,str,f64,str,i64,str,str,f64,i64,str,str,datetime[μs],bool,bool,bool
0,0,2004,12,10,"""20:45""",-100.0,"""Online Transaction""",7501849281341469857,"""ONLINE""",null,null,4722,null,"""No""",2004-12-10 20:45:00,true,false,true
0,0,2010,10,8,"""18:00""",-393.0,"""Online Transaction""",333722291367506728,"""ONLINE""",null,null,4722,null,"""No""",2010-10-08 18:00:00,true,false,true
0,0,2011,12,19,"""12:50""",-443.0,"""Online Transaction""",333722291367506728,"""ONLINE""",null,null,4722,null,"""No""",2011-12-19 12:50:00,true,false,true
0,0,2015,11,20,"""07:42""",-473.0,"""Online Transaction""",-8566951830324093739,"""ONLINE""",null,null,3640,null,"""Yes""",2015-11-20 07:42:00,true,false,true
0,2,2017,2,21,"""08:03""",-461.0,"""Online Transaction""",7501849281341469857,"""ONLINE""",null,null,4722,null,"""No""",2017-02-21 08:03:00,true,false,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1998,0,2019,2,27,"""15:09""",-441.0,"""Online Transaction""",3694722044710185708,"""ONLINE""",null,null,4722,null,"""No""",2019-02-27 15:09:00,true,false,true
1998,0,2019,11,27,"""11:08""",-354.0,"""Online Transaction""",3694722044710185708,"""ONLINE""",null,null,4722,null,"""No""",2019-11-27 11:08:00,true,false,true
1998,0,2019,12,29,"""19:40""",-122.0,"""Online Transaction""",3694722044710185708,"""ONLINE""",null,null,4722,null,"""No""",2019-12-29 19:40:00,true,false,true


### Per-user aggregation

The code below performs a user-level aggregation, collapsing millions of individual transaction records into a concise summary of spending behavior for each unique customer. By using `group_by("User")`, you are instructing Polars to organize the data into buckets based on the individual user ID. The `.agg()` function then calculates three key metrics for each bucket:

1. the total volume of money spent,
2. the average cost per purchase, and
3. the total count of transactions (using `pl.len()`).


In [15]:
user_stats = (
    df_polars.group_by("User")
    .agg(
        [
            pl.col("Amount").sum().alias("total_spent"),
            pl.col("Amount").mean().alias("average_amount"),
            pl.len().alias("num_transactions"),
        ]
    )
    .sort("User")
)

user_stats

User,total_spent,average_amount,num_transactions
i64,f64,f64,u32
0,1.6230e6,81.299989,19963
1,723491.89,81.11805,8919
2,1.4759e6,35.159687,41978
3,1.1865e6,117.277603,10117
4,1.7988e6,97.011698,18542
…,…,…,…
1995,348906.07,23.714135,14713
1996,739508.19,41.056417,18012
1997,705171.08,32.636233,21607


Rolling / window features (SQL-like mental model):


### 10 most recent transactions rolling sum

Window features depend on row order, so we first sort each user's transactions by time. Two details matter on a table this size:

- **Ties.** `"Datetime"` has minute resolution, and a user can make several transactions in the same minute. If you sort by `"User"` and `"Datetime"` alone, tied rows can come out in a different order on each run (and in a different order in another engine), and so can any feature that depends on order. We break ties by `"Card"` and then by `row_nr`, the row's position in the source file, so the order is the same every time.
- **Memory.** Sorting copies every column it carries. We keep only the columns the window features need, and turn `"Is Fraud?"` into a boolean `is_fraud` column on the way.

The second step calculates a rolling sum of the `"Amount"` column over the last 10 transactions for each user. By using `.over("User")`, you ensure that the rolling calculation is performed separately for each individual customer. The `.rolling_sum(window_size=10)` function then computes the sum of the last 10 transaction amounts, providing insight into recent spending trends.

In [16]:
df_sorted = (
    df_polars.with_row_index("row_nr")
    .select(
        "row_nr",
        "User",
        "Card",
        "Datetime",
        "Amount",
        "is_online",
        (pl.col("Is Fraud?") == "Yes").alias("is_fraud"),
    )
    .sort(["User", "Datetime", "Card", "row_nr"])
)

df_rolling = df_sorted.with_columns(
    pl.col("Amount")
    .rolling_sum(window_size=10)
    .over("User")
    .alias("rolling_amount_sum_10")
)

df_rolling.select(["User", "Card", "Datetime", "Amount", "rolling_amount_sum_10"]).head(12)

User,Card,Datetime,Amount,rolling_amount_sum_10
i64,i64,datetime[μs],f64,f64
0,0,2002-09-01 06:21:00,134.09,null
0,0,2002-09-01 06:42:00,38.48,null
0,0,2002-09-02 06:22:00,120.34,null
0,0,2002-09-02 17:45:00,128.95,null
0,0,2002-09-03 06:23:00,104.71,null
…,…,…,…,…
0,0,2002-09-04 06:09:00,123.5,null
0,0,2002-09-05 06:14:00,61.72,null
0,0,2002-09-05 09:35:00,57.1,948.92


The `.over("User")` part is the direct counterpart to SQL's `PARTITION BY User`. It ensures the rolling calculation "resets" or stays contained within each specific user's history, preventing a transaction from User A from being included in the sum for User B. In SQL terms, this operation is essentially:

```sql
SUM(Amount) OVER (
    PARTITION BY User
    ORDER BY Datetime, Card, row_nr
    ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
)
```

There is one difference. Polars returns `null` until a user has 10 transactions, which is why the first rows above are empty. The SQL window sums whatever rows are available, so it returns partial sums for a user's first nine transactions. Pass `min_samples=1` to `rolling_sum()` if you want the SQL behaviour.

This technique can be used in real-time fraud detection. By tracking a "rolling sum," you can detect "velocity attacks" where a card is used for many large purchases in a very short window. If this rolling sum spikes suddenly compared to the user's historical average, it provides a much stronger signal for fraud than any single transaction could on its own.

## Additional Analyses

The cells below extend the basic preprocessing pipeline with a few more
business-relevant analyses. Each one is something a fraud or marketing analyst
would routinely build on this dataset.


### Top Merchants by Total Spend

Knowing where your customers spend the most is the starting point for any
co-marketing or interchange-cost optimisation conversation.


In [17]:
top_merchants = (
    df_polars.group_by("Merchant Name")
    .agg(
        pl.col("Amount").sum().round(2).alias("total_spend"),
        pl.len().alias("n_transactions"),
        pl.col("Amount").mean().round(2).alias("avg_ticket"),
    )
    .sort("total_spend", descending=True)
    .head(10)
)
top_merchants

Merchant Name,total_spend,n_transactions,avg_ticket
i64,f64,u32,f64
-4282466774399734331,1.0179e8,1129061,90.15
1913477460590765860,3.5349e7,562158,62.88
-2088492411650162548,2.5555e7,720615,35.46
2027553650310142703,2.2045e7,1028485,21.43
4241336128694185533,1.9183e7,173206,110.75
-5467922351692495955,1.7097e7,374905,45.6
1799189980464955940,1.6418e7,1130230,14.53
-5162038175624867091,1.4271e7,676694,21.09
97032797689821735,1.4246e7,541794,26.29


### Spend Patterns by Hour of Day

A simple hour-of-day breakdown is often the first signal an analyst examines
when looking for unusual transaction timing (a common fraud heuristic).


In [18]:
hourly = (
    df_polars.with_columns(pl.col("Datetime").dt.hour().alias("hour"))
    .group_by("hour")
    .agg(
        pl.len().alias("n_transactions"),
        pl.col("Amount").sum().round(2).alias("total_spend"),
    )
    .sort("hour")
)
hourly

hour,n_transactions,total_spend
i8,u32,f64
0,235608,1.2522e7
1,213411,1.1973e7
2,214075,1.0928e7
3,183370,9692236.4
4,203550,1.1401e7
…,…,…
19,847344,4.3778e7
20,793351,4.1598e7
21,765024,3.8863e7


:::{tip} Visualising the result
Pipe the result into `.to_pandas()` and pass it to **Plotly Express** (with the
`simple_white` template) to get a publication-quality bar chart in one line:

```python
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

px.bar(hourly.to_pandas(), x="hour", y="n_transactions",
       title="Transactions by hour of day")
```

:::


### Fraud Rate by Use-Chip Category

`Use Chip` is a categorical column with three values: chip, swipe, and online.
Online transactions are well known to carry a higher fraud rate. Let's verify
that on this dataset.


In [19]:
fraud_by_channel = (
    df_polars.with_columns((pl.col("Is Fraud?") == "Yes").alias("is_fraud"))
    .group_by("Use Chip")
    .agg(
        pl.len().alias("n_transactions"),
        pl.col("is_fraud").sum().alias("n_fraud"),
        (pl.col("is_fraud").mean() * 100).round(3).alias("fraud_rate_pct"),
    )
    .sort("fraud_rate_pct", descending=True)
)
fraud_by_channel

Use Chip,n_transactions,n_fraud,fraud_rate_pct
str,u32,u32,f64
"""Online Transaction""",2713220,18349,0.676
"""Chip Transaction""",6287598,4836,0.077
"""Swipe Transaction""",15386082,6572,0.043


### Time Since Previous Transaction (per User)

Sequential timing features are often useful in fraud models. The expression below computes the gap between consecutive transactions for each user, using the sorted `df_sorted` from above. A burst of transactions only minutes apart is a classic account-takeover signal.

Because `"Datetime"` has minute resolution, every gap is a multiple of 60 seconds. A gap of 0 means the transaction happened in the same minute as the user's previous one.

In [20]:
df_with_gap = df_sorted.with_columns(
    (pl.col("Datetime") - pl.col("Datetime").shift(1).over("User"))
    .dt.total_seconds()
    .alias("seconds_since_last_txn")
)

df_with_gap.select(["User", "Datetime", "Amount", "seconds_since_last_txn"]).head(15)

User,Datetime,Amount,seconds_since_last_txn
i64,datetime[μs],f64,i64
0,2002-09-01 06:21:00,134.09,null
0,2002-09-01 06:42:00,38.48,1260
0,2002-09-02 06:22:00,120.34,85200
0,2002-09-02 17:45:00,128.95,40980
0,2002-09-03 06:23:00,104.71,45480
…,…,…,…
0,2002-09-05 20:18:00,76.07,38580
0,2002-09-05 20:41:00,53.91,1380
0,2002-09-06 06:16:00,110.37,34500


:::{important} `.over("User")` is Polars' `PARTITION BY`
`shift(1).over("User")` shifts each row by one **within a user's history**, so
the very first transaction for each user gets `null`. `shift` follows the
current row order, which is why we sorted first. This is the same as
`LAG(Datetime) OVER (PARTITION BY User ORDER BY Datetime, Card, row_nr)` in SQL.
:::

### A Composite Suspicious-Transaction Flag

Real fraud rules are usually combinations of several conditions. The example below flags a transaction when all three of these hold:

1. It is an online transaction.
2. Its amount is above the user's average amount **so far**, that is, the average over the user's earlier transactions only.
3. It happened in the same minute as the user's previous transaction (`seconds_since_last_txn == 0`).

The second condition uses an expanding (past-only) mean. A rule that runs when a transaction arrives can only know the past. If we averaged over the user's whole history, later transactions would leak into the rule, and it would look better on historical data than it could in production. `cum_sum().shift(1)` adds up the user's earlier amounts, and `int_range(pl.len())` counts them (0 for the user's first transaction).

In [21]:
flagged = df_with_gap.with_columns(
    (
        pl.col("Amount").cum_sum().shift(1).over("User")
        / pl.int_range(pl.len()).over("User")
    ).alias("user_prior_avg_amount")
).with_columns(
    (
        pl.col("is_online")
        & (pl.col("Amount") > pl.col("user_prior_avg_amount"))
        & (pl.col("seconds_since_last_txn") == 0)
    )
    # A user's first transaction has no history, so the rule cannot fire.
    .fill_null(False)
    .alias("is_suspicious")
)

(
    flagged.group_by("is_suspicious")
    .agg(
        pl.len().alias("n_transactions"),
        pl.col("Amount").mean().round(2).alias("avg_amount"),
    )
    .sort("is_suspicious")
)

is_suspicious,n_transactions,avg_amount
bool,u32,f64
false,24383426,43.62
true,3474,117.25


### How good is the rule?

The dataset has a fraud label, so we can check the rule against it. Two numbers summarise a screening rule:

- **Precision:** of the transactions the rule flags, the share that are fraud.
- **Recall:** of all fraud transactions, the share the rule flags.

Compare precision with the fraud rate across all transactions. A useful rule should flag fraud far more often than picking transactions at random would.

In [22]:
n_flagged = flagged["is_suspicious"].sum()
n_fraud = flagged["is_fraud"].sum()
n_flagged_fraud = (flagged["is_suspicious"] & flagged["is_fraud"]).sum()

precision = n_flagged_fraud / n_flagged
recall = n_flagged_fraud / n_fraud
fraud_rate = flagged["is_fraud"].mean()

print(f"Flagged transactions:        {n_flagged:,}")
print(f"Flagged and labelled fraud:  {n_flagged_fraud:,}")
print(f"All fraud transactions:      {n_fraud:,}")
print(f"Precision:                   {precision:.1%}")
print(f"Recall:                      {recall:.2%}")
print(f"Fraud rate, all rows:        {fraud_rate:.2%}")

Flagged transactions:        3,474
Flagged and labelled fraud:  119
All fraud transactions:      29,757
Precision:                   3.4%
Recall:                      0.40%
Fraud rate, all rows:        0.12%


The rule flags about 3,500 transactions, and about 3% of them are labelled fraud. That is nearly 30 times the fraud rate across all transactions (0.12%) and about five times the fraud rate of online transactions alone (0.68%, from the table above), so the rule does concentrate fraud. But its recall is well under 1%: it catches only about 120 of the nearly 30,000 fraud cases. Almost all fraud in this dataset does not look like a burst of above-average online purchases in the same minute.

This is typical of a single hand-written rule. It can be a reasonable way to send a short list to a human reviewer, but it misses most of the problem. Raising recall means adding more signals (amount, merchant category, location, time of day, error codes), and at that point a model trained on the label is usually the better tool.

:::{caution} Heuristic flags ≠ a fraud model
A composite rule like the one above is a useful **screening filter**: it can
cut a 24-million-row dataset down to a few thousand "interesting" rows for a
human analyst. As the precision and recall above show, it is not a substitute
for a calibrated probabilistic model trained on labeled fraud examples.
:::

## Summary

In this notebook you:

- Read a 2.3 GB CSV with about 24.4 million transactions into Polars.
- Found that `"Merchant State"`, `"Zip"`, and `"Errors?"` are the only columns with missing values, and that the nulls are meaningful (online merchants, merchants outside the U.S., and transactions without errors), so we kept them.
- Converted `"Amount"` from a `$` string to a number, combined the date parts into a minute-resolution `"Datetime"`, and saved the cleaned table as Parquet for the DuckDB notebook.
- Built flag columns, per-user totals, top merchants, and hourly patterns. Online transactions have a fraud rate of about 0.68%, compared with about 0.08% for chip and 0.04% for swipe transactions.
- Built window features with a deterministic sort order: a rolling 10-transaction sum, the time since the previous transaction, and an expanding (past-only) average.
- Tested a composite fraud rule against the label. It flags about 3,500 transactions with a precision of about 3% and a recall well under 1%.

The DuckDB notebook repeats these steps in SQL and checks that both engines flag the same rows.